# 06 - Google Trends Extraction (Pytrends)

### Objective
This notebook handles the automated extraction of historical search interest data using the unofficial Google Trends API (`pytrends`).

### Methodology: The Anchor search_term
Google Trends returns relative indices (0-100) rather than absolute volumes. To ensure that the trends of all destinations in our long-tail are directly comparable within a single time-series model, we use a constant **Anchor Keyword** across all API requests.

**Mathematical Justification for the Anchor Choice:**
The selection of this keyword derives from the analysis of absolute search volumes extracted in the previous notebook (05_DataForSeo_volume_searches). To prevent the anchor term from "flattening" low-volume destinations (if it is too large) or disappearing against top destinations (if it is too small), it was optimized based on two statistical metrics:

1. **Volume (75th-95th Percentiles):** We filtered for medium-high volume destinations to act as an effective connecting bridge.
2. **Stability (Coefficient of Variation, $CV = \sigma / \mu$):** We searched for the destination with the lowest annual variance to avoid seasonal distortions during normalization.

**Final Selection:** `"viaje a argentina"`
Although high-volume destinations like Norway appeared in the initial analysis, they exhibited bimodal distributions (sharp peaks in summer and winter). Argentina proved to be the most statistically robust candidate:
- Average monthly volume: ~3,000 searches (Sufficient relative weight).
- Coefficient of Variation: 0.279 (Stable and flat demand throughout the year, making it an ideal continuous measurement baseline).

**Known limitation:** Cross-batch index comparability is not fully guaranteed due to Google Trend's inherent instability between API calls. 
This is a structural limitation of the unofficial API, not a methodological error. It is partially mitigated by mean aggregation during consolidation.

## 1. Imports

In [2]:
import pandas as pd
from pathlib import Path
from pytrends.request import TrendReq
import time
import random

## 2. Paths & Constants

In [3]:
PROCESSED_PATH = Path("../data/processed")
INTERIM_PATH = Path("../data/interim")
OUTPUT_PATH = Path("../outputs/keywords")

INTERIM_PATH.mkdir(parents=True, exist_ok=True)


ANCHOR_KEYWORD = "viaje a argentina"
TIMEFRAME = '2018-07-01 2026-03-31'
GEO_LOCATION = 'ES'

## 3. Data Ingestion
Loading the prefixed keywords generated in the previous notebook.

In [4]:
df_keywords = pd.read_parquet(OUTPUT_PATH / "04_keywords_with_prefix.parquet")

# Extract the unique search queries
keywords_to_fetch = df_keywords["google_ads_keyword"].dropna().unique().tolist() 

print(f"✔ Total unique queries to fetch: {len(keywords_to_fetch)}")
print(f"Sample: {keywords_to_fetch[:4]}")

✔ Total unique queries to fetch: 220
Sample: ['viaje a albania', 'viaje a balcanes', 'viaje a alemania', 'viaje a andorra']


## 4. API Initialization & Core Functions

#### Initialize Pytrends (Spanish localization, Central European Time offset)

In [ ]:
# Sort by priority and drop duplicates keeping the highest priority source
df_keyword_mapping = (
    df_keyword_mapping
    .sort_values("priority")
    .drop_duplicates(
        subset=["search_term", "parent_country"],
        keep="first"
    )
    .drop(columns="priority")
)

print(f"Records before dedup: {records_before_dedup}")
print(f"Records after dedup: {len(df_keyword_mapping)}")
print(f"Dropped: {records_before_dedup - len(df_keyword_mapping)}")

Records before dedup: 395
Records after dedup: 236
Dropped: 159


In [5]:
pytrends = TrendReq(
    hl="es-ES",
    tz=60
)

#### Normalization Note
- Rescaling is applied per-batch independently. Cross-batch comparability is not fully guaranteed due to GT's inherent API instability — the same anchor may return slightly different max values across calls.
- This is a structural limitation of Google Trends, not a methodological error. It is partially mitigated by mean aggregation in Section 6.

In [5]:
def get_normalized_trends(keywords_batch):
    """
    Fetches Google Trends data, calibrates it mathematically using the anchor, 
    and unpivots the result to a long format.
    """
    
    pytrends.build_payload(
        kw_list=keywords_batch,
        timeframe=TIMEFRAME,
        geo=GEO_LOCATION
    )

    df = pytrends.interest_over_time()

    if df.empty:
        return None
        
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])

    anchor = keywords_batch[0] 
    
    if anchor in df.columns:
        anchor_max = df[anchor].max()     
        
        if anchor_max > 0:
            multiplier_factor = 100.0 / anchor_max
            
            for column in keywords_batch:
                if column in df.columns:
                    df[column] = df[column] * multiplier_factor
        else:
            print(f"Warning: The anchor has a maximum of 0 in this batch: {keywords_batch}")
            return None

    # Changes the df from wide-format to long-format
    df_long = df.reset_index().melt(
        id_vars="date",
        var_name="search_term",
        value_name="trend_index"
    )

    df_long = df_long.rename(columns={"date": "period"})
    
    return df_long

## 5. Extraction Engine with Checkpoints
To respect rate limits and avoid 429 errors from Google, this loop incorporates random sleep intervals and saves partial progress (checkpoints) locally.

In [6]:
def force_prefix(word):
    word = str(word).strip().lower()
    if not word.startswith("viaje a "):
        return f"viaje a {word}"
    return word

In [ ]:
CHECKPOINT_FILE = (
    INTERIM_PATH /
    "06_google_trends_anchor_checkpoint.parquet"
)

# Identify missing queries

# Load existing progress to avoid redundant API calls
if CHECKPOINT_FILE.exists():
    df_checkpoint = pd.read_parquet(CHECKPOINT_FILE)
    
    unique_words_in_file = df_checkpoint["search_term"].unique()
    
    # set for clean words
    downloaded_queries = set()
    
    for word in unique_words_in_file:
        clean_word = force_prefix(word)
        downloaded_queries.add(clean_word)
        
    print(f"✔ Checkpoint loaded. Previously fetched queries: {len(downloaded_queries)}")

else:
    # If no file, empty variables
    df_checkpoint = pd.DataFrame(columns=["period", "search_term", "trend_index"])
    downloaded_queries = set()
    print("No checkpoint found. Starting new extraction.")


# 3. Apply the rule to the original list of words
keywords_to_fetch_net = []

# We go word by word, clean it, and append it to our new list
for word in keywords_to_fetch:
    clean_word = force_prefix(word)
    keywords_to_fetch_net.append(clean_word)


# 4. Calculate exactly which words are missing
missing_queries = []

# We check every word in our clean target list
for word in keywords_to_fetch_net:
    # If word NOT in the history,added to missing list
    if word not in downloaded_queries:
        missing_queries.append(word)


# remove anchor from pending list
if ANCHOR_KEYWORD in missing_queries:
    missing_queries.remove(ANCHOR_KEYWORD)

print(f"Queries pending to extract: {len(missing_queries)}")

✔ Checkpoint loaded. Previously fetched queries: 2
Queries pending to extract: 218


In [7]:
RUN_TRENDS_DOWNLOAD = False # Change this variable to True to execute the API extraction loop

In [ ]:
if RUN_TRENDS_DOWNLOAD:

    for keyword in missing_queries:
        if keyword == ANCHOR_KEYWORD:
            continue

        batch = [
            ANCHOR_KEYWORD,
            keyword
        ]

        try:
            print(f"Downloading: {batch}")
            df_temp = get_normalized_trends(
                batch
            )

            if df_temp is not None:
                if df_checkpoint.empty:
                    df_checkpoint = df_temp.copy()
                else:
                    df_checkpoint = pd.concat([df_checkpoint, df_temp], ignore_index=True)                
                df_checkpoint["period"] = pd.to_datetime(df_checkpoint["period"], format="mixed")
                df_checkpoint.to_parquet(CHECKPOINT_FILE)
                print("✔ Checkpoint guardado")

            wait_time = random.randint(45,80)

            print(
                f"Esperando {wait_time} segundos...\n"
            )

            time.sleep(wait_time)

        except Exception as e:
            print(f"Error {e}")
            print("Batch saltado. Pausa de seguridad")
            time.sleep(200)


Downloading: ['viaje a argentina', 'viaje a balcanes']
✔ Checkpoint guardado
Esperando 50 segundos...

Downloading: ['viaje a argentina', 'viaje a alemania']
✔ Checkpoint guardado
Esperando 65 segundos...

Downloading: ['viaje a argentina', 'viaje a andorra']
✔ Checkpoint guardado
Esperando 70 segundos...

Downloading: ['viaje a argentina', 'viaje a angola']
✔ Checkpoint guardado
Esperando 74 segundos...

Downloading: ['viaje a argentina', 'viaje a arabia saudi']
✔ Checkpoint guardado
Esperando 68 segundos...

Downloading: ['viaje a argentina', 'viaje a arabia saudita']
✔ Checkpoint guardado
Esperando 76 segundos...

Downloading: ['viaje a argentina', 'viaje a argelia']
✔ Checkpoint guardado
Esperando 50 segundos...

Downloading: ['viaje a argentina', 'viaje a crucero australis']
✔ Checkpoint guardado
Esperando 52 segundos...

Downloading: ['viaje a argentina', 'viaje a patagonia']
✔ Checkpoint guardado
Esperando 59 segundos...

Downloading: ['viaje a argentina', 'viaje a armenia']
✔ C

## 6. Data Quality

In [ ]:
print("--- DATA QUALITY & CONSOLIDATION ---")

# Load the checkpoint
df_final = pd.read_parquet(CHECKPOINT_FILE)
print(f"Original Shape: {df_final.shape}")

# Remove prefix
df_final["search_term"] = (
    df_final["search_term"]
    .str.replace("viaje a ", "", regex=False)
    .str.strip()
)


df_final["period"] = pd.to_datetime(
    df_final["period"], 
    format="mixed"
)

# Check for duplicates before agg
duplicates_count = df_final[["period", "search_term"]].duplicated().sum()
print(f"Repeated period-search_term combinations: {duplicates_count}")

# Aggregate overlapping Anchor data points
df_final = (
    df_final
    .groupby(["period", "search_term"], as_index=False)
    .agg({"trend_index": "mean"})
)

# Final Validation
print(f"\nFinal Consolidated Shape: {df_final.shape}")
print(f"Unique Destinations: {df_final['search_term'].nunique()}")

# Export the modeling-ready dataset
FINAL_OUTPUT = PROCESSED_PATH / "06_google_trends_master_v2.parquet"
df_final.to_parquet(FINAL_OUTPUT)

print(f"\n✔ Final Time-Series Dataset exported to: {FINAL_OUTPUT}")
display(df_final.head())

--- DATA QUALITY & CONSOLIDATION ---
Original Shape: (39445, 3)
Repeated period-search_term combinations: 17899

Final Consolidated Shape: (21546, 3)
Unique Destinations: 220

✔ Final Time-Series Dataset exported to: ..\data\processed\05_google_trends_master_v2.parquet


,period,search_term,trend_index
0,2018-01-01,abu dhabi,0.0
1,2018-01-01,albania,0.0
2,2018-01-01,alemania,14.0
3,2018-01-01,andorra,18.0
4,2018-01-01,angola,0.0
